<a href="https://colab.research.google.com/github/priyu9-star/BudgetWise-AI-based-Expense-Forecasting-Tool-Batch-6-Team-C-/blob/main/some_ui_part.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================
# SPENDGENIE — Single Colab cell (UI A: Notion-like, collapsible sidebar)
# ALL PAGES: Expense Tracker, Dashboard, Prediction, Expense Input, Profile, Logout
# Data columns expected: item (category), price (amount), date (date)
# ======================================================

!pip install -q flask pyngrok pandas

import pandas as pd
from flask import Flask, render_template_string, request, redirect, url_for
from pyngrok import ngrok
from datetime import datetime

app = Flask(__name__)

# -------------------------
# In-memory storage (global)
# -------------------------
main_df = pd.DataFrame(columns=["item", "price", "date", "description"])
saved_expenses = []              # human-friendly list of manual additions
available_categories = []        # derived from main_df

# -------------------------
# Helpers
# -------------------------
def normalize_main_df():
    """Ensure main_df has correct dtypes and update available_categories."""
    global main_df, available_categories
    if main_df is None or main_df.empty:
        main_df = pd.DataFrame(columns=["item", "price", "date", "description"])
    for c in ["item", "price", "date", "description"]:
        if c not in main_df.columns:
            main_df[c] = None
    # price numeric
    main_df["price"] = pd.to_numeric(main_df["price"], errors="coerce").fillna(0.0)
    # parse dates robustly
    def parse_date(x):
        if pd.isna(x) or x == "":
            return pd.NaT
        if isinstance(x, (pd.Timestamp, datetime)):
            return pd.to_datetime(x)
        try:
            return pd.to_datetime(x)
        except:
            for fmt in ("%Y-%m-%d","%d-%m-%Y","%d/%m/%Y","%m/%d/%Y"):
                try:
                    return pd.to_datetime(x, format=fmt)
                except:
                    pass
            return pd.NaT
    main_df["date"] = main_df["date"].apply(parse_date)
    available_categories = sorted(main_df["item"].dropna().astype(str).unique().tolist())

def totals_by_category(period="all"):
    """Return dict category -> total for the given period (all,daily,weekly,monthly)."""
    normalize_main_df()
    df = main_df.copy()
    if df.empty:
        return {}
    df = df.dropna(subset=["item","price"]).copy()
    if period == "daily":
        today = pd.Timestamp(datetime.now().date())
        df = df[df["date"].dt.date == today.date()]
    elif period == "weekly":
        today = pd.Timestamp(datetime.now().date())
        week_ago = today - pd.Timedelta(days=6)  # 7-day window
        df = df[df["date"].dt.date >= week_ago.date()]
    elif period == "monthly":
        today = pd.Timestamp(datetime.now().date())
        month_ago = today - pd.Timedelta(days=29)  # 30-day window
        df = df[df["date"].dt.date >= month_ago.date()]
    grouped = df.groupby("item")["price"].sum().to_dict()
    # ensure keys for all known categories
    for cat in available_categories:
        grouped.setdefault(cat, 0.0)
    return grouped

def total_expense(period="all"):
    return sum(totals_by_category(period).values())

def avg_per_day_for_category(cat):
    normalize_main_df()
    df = main_df.copy()
    df = df[df["item"] == cat].dropna(subset=["price","date"]).copy()
    if df.empty:
        return 0.0
    total = df["price"].sum()
    min_date = df["date"].min().date()
    max_date = df["date"].max().date()
    span_days = (max_date - min_date).days + 1
    if span_days <= 0:
        span_days = 1
    return float(total / span_days)

# -------------------------
# UI A (collapsible sidebar) BASE TEMPLATE
# Note: we keep visual markup identical to your provided UI A, but
# expose a {{ content }} block where each route injects page-specific HTML.
# -------------------------
BASE_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>SpendGenie</title>
<style>
    body {
        margin: 0;
        background: #0f0f0f;
        color: white;
        font-family: Inter, sans-serif;
        display: flex;
    }
    .sidebar {
        width: 240px;
        background: #151515;
        height: 100vh;
        border-right: 1px solid #222;
        padding-top: 20px;
        position: fixed;
        top: 0;
        left: 0;
        transition: width 0.3s ease;
        overflow: hidden;
    }
    .sidebar.collapsed { width: 70px; }
    .toggle-btn {
        position: absolute;
        top: 15px;
        right: -15px;
        width: 30px;
        height: 30px;
        background: #0ea5ff;
        border-radius: 50%;
        color: #fff;
        font-size: 16px;
        border: none;
        cursor: pointer;
    }
    .sidebar-title {
        font-size: 22px;
        font-weight: 700;
        margin-left: 25px;
        margin-bottom: 35px;
        opacity: 1;
        transition: opacity 0.3s ease;
        display: flex;
        align-items: center;
        gap: 8px;
    }
    .sidebar.collapsed .sidebar-title { opacity: 0; }
    .menu-item {
        padding: 12px 20px;
        display: flex;
        align-items: center;
        gap: 12px;
        font-size: 16px;
        cursor: pointer;
        border-radius: 10px;
        margin: 8px 10px;
        color: #ddd;
        white-space: nowrap;
        text-decoration: none;
    }
    .menu-item:hover { background: #262626; }
    .menu-item.active { background: #0ea5ff; color: white; }
    .menu-label { transition: opacity 0.3s ease; }
    .sidebar.collapsed .menu-label { opacity: 0; }
    .icon { font-size: 18px; }
    .sidebar-bottom { position: absolute; bottom: 25px; width: 100%; }
    header {
        position: fixed;
        left: 240px;
        width: calc(100% - 240px);
        background: linear-gradient(90deg, #022238, #043a63);
        padding: 18px 40px;
        display: flex;
        justify-content: space-between;
        align-items: center;
        transition: left 0.3s ease, width 0.3s ease;
    }
    .sidebar.collapsed + header { left: 70px; width: calc(100% - 70px); }
    .logo { font-size: 22px; font-weight: 700; }
    .main {
        margin-left: 240px;
        margin-top: 90px;
        padding: 30px 50px;
        width: calc(100% - 240px);
        transition: margin-left 0.3s ease, width 0.3s ease;
    }
    .sidebar.collapsed + header + .main { margin-left: 70px; width: calc(100% - 70px); }
    .upload-box {
        background: #1a1a1a;
        padding: 28px;
        border-radius: 14px;
        border: 1px solid #2b2b2b;
        text-align: center;
    }
    input[type=file] { background: #111; color: white; border: 1px solid #444; padding: 10px; width: 100%; border-radius: 10px; }
    button { background: #0ea5ff; color: white; padding: 10px 22px; margin-top: 10px; border: none; border-radius: 8px; cursor: pointer; }
    .grid { margin-top: 25px; display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 20px; }
    .card { background: #1c1c1c; padding: 22px; border-radius: 14px; border: 1px solid #2d2d2d; }
    .cat-title { font-size: 20px; font-weight: 700; }
    .cat-amount { margin-top: 5px; color: #c9c9c9; }
</style>

<script>
function toggleSidebar() {
    document.querySelector(".sidebar").classList.toggle("collapsed");
}
</script>
</head>
<body>

<!-- SIDEBAR -->
<div class="sidebar" id="side">
    <button class="toggle-btn" onclick="toggleSidebar()">≡</button>

    <div class="sidebar-title">⚡ SpendGenie</div>

    <a class="menu-item {% if active=='home' %}active{% endif %}" href="/"><span class="icon">📁</span><span class="menu-label" style="margin-left:6px;">Expense Tracker</span></a>
    <a class="menu-item {% if active=='dashboard' %}active{% endif %}" href="/dashboard"><span class="icon">📊</span><span class="menu-label" style="margin-left:6px;">Dashboard</span></a>
    <a class="menu-item {% if active=='prediction' %}active{% endif %}" href="/prediction"><span class="icon">🤖</span><span class="menu-label" style="margin-left:6px;">Prediction</span></a>
    <a class="menu-item {% if active=='expense_input' %}active{% endif %}" href="/expense_input"><span class="icon">📝</span><span class="menu-label" style="margin-left:6px;">Expense Input</span></a>

    <div class="sidebar-bottom">
        <a class="menu-item {% if active=='profile' %}active{% endif %}" href="/profile"><span class="icon">👤</span><span class="menu-label" style="margin-left:6px;">Profile</span></a>
        <a class="menu-item {% if active=='logout' %}active{% endif %}" href="/logout"><span class="icon">🚪</span><span class="menu-label" style="margin-left:6px;">Logout</span></a>
    </div>
</div>

<!-- HEADER -->
<header>
    <div class="logo">⚡ SpendGenie</div>
    <div>Upload | Logout</div>
</header>

<!-- MAIN CONTENT (page-specific) -->
<div class="main">
    {{ content }}
</div>

</body>
</html>
"""

# -------------------------
# Routes
# -------------------------

@app.route("/", methods=["GET","POST"])
def home():
    """Expense Tracker page (upload + category grid + view filter)."""
    global main_df, available_categories
    uploaded = False
    if request.method == "POST":
        file = request.files.get("file")
        if file:
            # read csv and attempt to map columns case-insensitively
            df = pd.read_csv(file)
            lower_map = {c.lower(): c for c in df.columns}
            mapped_item = lower_map.get("item") or lower_map.get("category") or df.columns[0]
            mapped_price = lower_map.get("price") or lower_map.get("amount") or (df.columns[1] if len(df.columns)>1 else mapped_item)
            mapped_date = lower_map.get("date") or (df.columns[2] if len(df.columns)>2 else None)

            newdf = pd.DataFrame()
            # item
            try:
                newdf["item"] = df[mapped_item].astype(str)
            except:
                newdf["item"] = df.iloc[:,0].astype(str)
            # price
            try:
                newdf["price"] = pd.to_numeric(df[mapped_price], errors="coerce").fillna(0.0)
            except:
                newdf["price"] = 0.0
            # date
            if mapped_date and mapped_date in df.columns:
                try:
                    newdf["date"] = pd.to_datetime(df[mapped_date], errors="coerce")
                except:
                    newdf["date"] = pd.NaT
            else:
                newdf["date"] = pd.NaT
            newdf["description"] = ""
            main_df = pd.concat([main_df, newdf], ignore_index=True, sort=False)
            normalize_main_df()
            uploaded = True
            # redirect to prediction to show generated page immediately
            return redirect(url_for("prediction"))

    totals = totals_by_category(period="all")
    total_all = total_expense(period="all")
    view = request.args.get("view","all")
    content = render_template_string("""
        <h1>Expense Tracker</h1>

        <div class="upload-box">
            <form method="POST" enctype="multipart/form-data">
                <input type="file" name="file" accept=".csv" required>
                <button type="submit">Upload CSV</button>
            </form>
            {% if uploaded %}
                <div style="margin-top:10px;color:#0ea5ff;">CSV uploaded successfully — categories loaded.</div>
            {% endif %}
        </div>

        <div style="margin-top:18px;display:flex;align-items:center;gap:12px;">
            <div class="card" style="padding:12px 18px;">
                <strong>Total:</strong> ₹{{'%.2f'|format(total_all)}}
            </div>

            <form method="GET" action="/" style="display:inline-block;">
                <label style="margin-left:8px;color:#bbb;">View:</label>
                <select name="view" onchange="this.form.submit()" style="margin-left:8px;padding:6px;border-radius:6px;background:#111;color:white;border:1px solid #333;">
                    <option value="all" {% if view=='all' %}selected{% endif %}>All</option>
                    <option value="daily" {% if view=='daily' %}selected{% endif %}>Daily</option>
                    <option value="weekly" {% if view=='weekly' %}selected{% endif %}>Weekly</option>
                    <option value="monthly" {% if view=='monthly' %}selected{% endif %}>Monthly</option>
                </select>
            </form>
        </div>

        {% if totals %}
        <h2 style="margin-top:20px;">Categories</h2>
        <div class="grid">
            {% for cat, amt in totals.items() %}
                <div class="card">
                    <div class="cat-title">{{cat}}</div>
                    <div class="cat-amount">₹{{'%.2f'|format(amt)}}</div>
                </div>
            {% endfor %}
        </div>
        {% endif %}
    """, totals=totals, total_all=total_all, uploaded=uploaded, view=view)
    return render_template_string(BASE_TEMPLATE, active="home", page_title="Expense Tracker", content=content)

@app.route("/dashboard")
def dashboard():
    """Simple dashboard showing totals with view filter."""
    view = request.args.get("view","all")
    totals = totals_by_category(period=view)
    total_all = total_expense(period=view)
    content = render_template_string("""
        <h1>Dashboard</h1>
        <div style="display:flex;align-items:center;justify-content:space-between;">
            <div class="card" style="padding:12px 18px;"><strong>Total:</strong> ₹{{'%.2f'|format(total_all)}}</div>
            <form method="GET" action="/dashboard" style="display:inline-block;">
                <select name="view" onchange="this.form.submit()" style="padding:6px;border-radius:6px;background:#111;color:white;border:1px solid #333;">
                    <option value="all" {% if view=='all' %}selected{% endif %}>All</option>
                    <option value="daily" {% if view=='daily' %}selected{% endif %}>Daily</option>
                    <option value="weekly" {% if view=='weekly' %}selected{% endif %}>Weekly</option>
                    <option value="monthly" {% if view=='monthly' %}selected{% endif %}>Monthly</option>
                </select>
            </form>
        </div>

        <div style="margin-top:18px;">
            <div class="grid">
                {% for cat, amt in totals.items() %}
                    <div class="card">
                        <div class="cat-title">{{cat}}</div>
                        <div class="cat-amount">₹{{'%.2f'|format(amt)}}</div>
                    </div>
                {% endfor %}
            </div>
        </div>

        <div style="margin-top:20px;">
            <h3>Recent manual additions</h3>
            {% if saved_expenses %}
                <ul>
                {% for e in saved_expenses[-8:] %}
                    <li>{{ e['date'] }} — {{ e['category'] }} — ₹{{ e['amount'] }} — {{ e['description'] }}</li>
                {% endfor %}
                </ul>
            {% else %}
                <div style="color:#999;">No manual expenses added yet.</div>
            {% endif %}
        </div>
    """, totals=totals, total_all=total_all, view=view, saved_expenses=saved_expenses)
    return render_template_string(BASE_TEMPLATE, active="dashboard", page_title="Dashboard", content=content)

@app.route("/prediction", methods=["GET","POST"])
def prediction():
    """Prediction page that shows category totals and allows category-specific prediction."""
    normalize_main_df()
    if not available_categories:
        content = "<div class='card'><p style='color:#ffbb00'>Please upload a CSV first (Expense Tracker). Prediction requires data.</p></div>"
        return render_template_string(BASE_TEMPLATE, active="prediction", page_title="Prediction", content=content)

    # category totals (all time)
    category_totals = main_df.groupby("item")["price"].sum().to_dict()

    # simple predicted next-month = mean(monthly sums)
    df = main_df.dropna(subset=["date"]).copy()
    if not df.empty:
        df["month"] = df["date"].dt.to_period("M")
        monthly_sum = df.groupby("month")["price"].sum()
        predicted_next_month = float(monthly_sum.mean()) if len(monthly_sum) > 0 else 0.0
    else:
        predicted_next_month = 0.0

    # category-specific prediction (POST)
    result = None
    explanation = None
    if request.method == "POST":
        cat = request.form.get("category")
        count = float(request.form.get("count", "1"))
        unit = request.form.get("unit", "days")
        days = count if unit == "days" else (count * 7 if unit == "weeks" else count * 30)
        df_cat = df[df["item"] == cat]
        if not df_cat.empty:
            total = df_cat["price"].sum()
            dmin = df_cat["date"].min()
            dmax = df_cat["date"].max()
            total_days = max((dmax - dmin).days + 1, 1)
            daily_avg = total / total_days
            prediction_val = daily_avg * days
            result = round(prediction_val, 2)
            explanation = f"Avg ₹{daily_avg:.2f}/day × {int(days)} days → ₹{result:.2f}"
        else:
            result = 0.0
            explanation = "No historical data for selected category."

    content = render_template_string("""
        <h1>Prediction</h1>

        <div class="card">
            <strong>Estimated Next Month (simple forecast):</strong>
            <div style="margin-top:8px;font-weight:700;">₹{{'%.2f'|format(predicted_next_month)}}</div>
        </div>

        <h2 style="margin-top:20px;">Category Breakdown</h2>
        <div class="grid">
            {% for c,a in category_totals.items() %}
            <div class="card">
                <div class="cat-title">{{c}}</div>
                <div class="cat-amount">₹{{'%.2f'|format(a)}}</div>
            </div>
            {% endfor %}
        </div>

        <h2 style="margin-top:28px;">Predict for a Category</h2>
        <div class="card" style="max-width:640px;">
            <form method="POST">
                <label>Category</label>
                <select name="category" style="width:100%;padding:10px;margin-top:8px;">
                    {% for c in cats %}<option>{{c}}</option>{% endfor %}
                </select>

                <div style="display:flex;gap:8px;margin-top:10px;">
                    <input type="number" name="count" min="1" value="1" style="padding:10px;width:120px;">
                    <select name="unit" style="padding:10px;">
                        <option value="days">Days</option>
                        <option value="weeks">Weeks</option>
                        <option value="months">Months</option>
                    </select>
                </div>

                <button type="submit" style="margin-top:12px;background:#0ea5ff;padding:10px;border:none;border-radius:8px;color:white;">Predict</button>

                {% if result is not none %}
                    <div style="margin-top:12px;color:#0ea5ff;font-weight:700;">Predicted: ₹{{ result }}</div>
                    <div style="color:#bbb;margin-top:6px;">{{ explanation }}</div>
                {% endif %}
            </form>
        </div>
    """, category_totals=category_totals, predicted_next_month=predicted_next_month, cats=available_categories, result=result, explanation=explanation)

    return render_template_string(BASE_TEMPLATE, active="prediction", page_title="Prediction", content=content)

@app.route("/expense_input", methods=["GET","POST"])
def expense_input():
    """Expense input page that uses dataset-derived categories and appends to main_df."""
    global main_df, saved_expenses, available_categories
    normalize_main_df()
    if not available_categories:
        content = "<div class='card'><p style='color:#ffbb00'>Please upload a CSV first (Expense Tracker). Expense Input requires dataset categories.</p></div>"
        return render_template_string(BASE_TEMPLATE, active="expense_input", page_title="Expense Input", content=content)

    success = False
    if request.method == "POST":
        cat = request.form.get("category")
        amt = request.form.get("amount")
        date_str = request.form.get("date")
        desc = request.form.get("description", "")
        try:
            amt_val = float(amt)
        except:
            amt_val = 0.0
        try:
            date_val = pd.to_datetime(date_str)
        except:
            date_val = pd.Timestamp(datetime.now())
        saved_expenses.append({"category": cat, "amount": amt_val, "date": date_val.strftime("%Y-%m-%d"), "description": desc})
        newrow = {"item": cat, "price": amt_val, "date": date_val, "description": desc}
        main_df = main_df.append(newrow, ignore_index=True)
        normalize_main_df()
        success = True

    content = render_template_string("""
        <h1>Expense Input</h1>

        <div class="card" style="max-width:680px;">
            <form method="POST">
                <label>Category</label>
                <select name="category" style="width:100%;padding:10px;margin-top:8px;">
                    {% for c in cats %}<option>{{c}}</option>{% endfor %}
                </select>

                <label style="margin-top:8px;">Amount</label>
                <input type="number" name="amount" step="0.01" style="width:100%;padding:10px;margin-top:8px;" required>

                <label style="margin-top:8px;">Date</label>
                <input type="date" name="date" style="width:100%;padding:10px;margin-top:8px;" required>

                <label style="margin-top:8px;">Description</label>
                <textarea name="description" style="width:100%;padding:10px;margin-top:8px;"></textarea>

                <button type="submit" style="margin-top:12px;background:#0ea5ff;padding:10px;border:none;border-radius:8px;color:white;">Save Expense</button>

                {% if success %}
                    <p style="color:#0ea5ff;margin-top:10px;">✔ Expense added successfully!</p>
                {% endif %}
            </form>
        </div>

        <div style="height:12px;"></div>

        <div class="card">
            <h4>Recent manual additions</h4>
            {% if saved_expenses %}
                <ul>
                    {% for e in saved_expenses[-10:] %}
                        <li>{{ e['date'] }} — {{ e['category'] }} — ₹{{ e['amount'] }} — {{ e['description'] }}</li>
                    {% endfor %}
                </ul>
            {% else %}
                <div style="color:#999;">No manual expenses yet.</div>
            {% endif %}
        </div>
    """, cats=available_categories, success=success, saved_expenses=saved_expenses)

    return render_template_string(BASE_TEMPLATE, active="expense_input", page_title="Expense Input", content=content)

@app.route("/profile")
def profile():
    content = "<h1>Profile</h1><div class='card'><p>Your profile information placeholder.</p></div>"
    return render_template_string(BASE_TEMPLATE, active="profile", page_title="Profile", content=content)

@app.route("/logout")
def logout():
    content = "<h1>Logout</h1><div class='card'><p>You have been logged out (placeholder).</p></div>"
    return render_template_string(BASE_TEMPLATE, active="logout", page_title="Logout", content=content)

# -------------------------
# Run server (ngrok)
# -------------------------
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN"  # <-- replace with your ngrok token before running

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
try:
    ngrok.kill()
except:
    pass

public_url = ngrok.connect(5000).public_url
print("🚀 SpendGenie running at:", public_url)

app.run(port=5000)


ERROR:pyngrok.process.ngrok:t=2025-11-27T19:39:37+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2025-11-27T19:39:37+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2025-11-27T19:39:37+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nY

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.